# 모두몰 성장 추이 분석 노트 — 시간/순위 분석 (customers, orders, products, order_items)

**과제**: 시간/순위 분석 질문 5개 + SQL(CTE, 윈도우 함수) + 인사이트

**환경**: 이전 과제(day3)와 동일하게 로컬 **DuckDB** 인메모리 테이블로 동일 스키마/데이터를 구성해 실행합니다.

**조건 체크리스트**
- WITH(CTE) 2개 이상 → 전 문항, Q5는 CTE 2개
- 순위 함수(RANK/ROW_NUMBER/DENSE_RANK) 최소 1회 → Q2(RANK), Q3(ROW_NUMBER), Q5(DENSE_RANK)
- LAG/LEAD 최소 1회 → Q3, Q4
- SUM() OVER 누적 최소 1회 → Q1(누적), Q5(파티션 합계)

## 0. 환경 설정 및 테이블 생성 (DuckDB) — day3와 동일한 데이터

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect(database=":memory:")

con.execute("""
CREATE OR REPLACE TABLE customers (
    customer_id STRING,
    name STRING,
    country STRING,
    signup_date DATE,
    grade STRING
);
""")
con.execute("""
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
""")

con.execute("""
CREATE OR REPLACE TABLE orders (
    order_id STRING,
    customer_id STRING,
    order_date DATE,
    status STRING,
    amount DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")

con.execute("""
CREATE OR REPLACE TABLE products (
    product_id STRING,
    product_name STRING,
    category STRING,
    price DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000);
""")

con.execute("""
CREATE OR REPLACE TABLE order_items (
    order_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO order_items VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000);
""")

print("테이블 생성 완료: customers, orders, products, order_items")


## Q1. 월별 누적 매출 추이는 어떻게 되는가? (CTE + `SUM() OVER` 누적)

- order_items와 orders를 조인해 월별 매출을 먼저 구하고(CTE)
- 그 위에 `SUM() OVER (ORDER BY 월)`로 누적 매출 계산
- **사용**: CTE, `SUM() OVER` 누적

In [ ]:
q1 = con.sql("""
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS order_month,
        SUM(oi.quantity * oi.unit_price) AS monthly_sales
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY 1
)
SELECT
    order_month,
    monthly_sales,
    SUM(monthly_sales) OVER (ORDER BY order_month) AS cumulative_sales
FROM monthly_revenue
ORDER BY order_month
""").df()
q1


> **인사이트**: 누적 매출은 계속 증가하지만 월별 매출 자체는 12월(526,000)에 11월(839,000) 대비 크게 꺾였다가 2024년 1월(850,000)에 다시 최고치를 찍음 — 12월 하락이 계절적 비수기 때문인지 특정 취소·반품(status) 때문인지는 추가 확인 필요.

## Q2. 카테고리별 매출 상위 3개 상품은? (CTE + `RANK()`)

- **사용**: CTE, `RANK()`

In [ ]:
q2 = con.sql("""
WITH product_sales AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
),
ranked AS (
    SELECT
        category,
        product_name,
        total_revenue,
        RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS revenue_rank
    FROM product_sales
)
SELECT *
FROM ranked
WHERE revenue_rank <= 3
ORDER BY category, revenue_rank
""").df()
q2


> **인사이트**: Sandals 카테고리는 1위 여름 샌들(551,000)과 2위 슬리퍼 샌들(530,000) 격차가 작아 두 상품이 비등하게 팔리는 반면, 다른 카테고리는 1위 상품이 2위보다 뚜렷하게 앞섬(예: Running 트레일 러너 361,000 vs 에어러너 239,000) — 카테고리별 상품 의존도가 다르다는 뜻.

## Q3. 고객별 재구매까지 걸린 기간은? (CTE + `ROW_NUMBER()` + `LAG()`)

- **사용**: CTE, `ROW_NUMBER()`, `LAG()`

In [ ]:
q3 = con.sql("""
WITH customer_orders AS (
    SELECT
        customer_id,
        order_id,
        order_date,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date) AS order_seq
    FROM orders
)
SELECT
    customer_id,
    order_id,
    order_date,
    order_seq,
    order_date - LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS days_since_prev_order
FROM customer_orders
ORDER BY customer_id, order_date
""").df()
q3


> **인사이트**: 재구매 간격은 짧게는 9일(C001)부터 길게는 122일(C003)까지 편차가 크고, C009·C011·C014·C015는 첫 주문 이후 재구매 기록이 아직 없음 — 이 고객들이 이탈 위험군인지, 데이터 수집 기간이 짧아서인지는 구분이 필요.

## Q4. 월별 매출은 전월 대비 얼마나 증감했는가? (CTE + `LAG()`)

- **사용**: CTE, `LAG()`

In [ ]:
q4 = con.sql("""
WITH monthly_revenue AS (
    SELECT
        DATE_TRUNC('month', o.order_date) AS order_month,
        SUM(oi.quantity * oi.unit_price) AS monthly_sales
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY 1
)
SELECT
    order_month,
    monthly_sales,
    LAG(monthly_sales) OVER (ORDER BY order_month) AS prev_month_sales,
    ROUND(
        (monthly_sales - LAG(monthly_sales) OVER (ORDER BY order_month))
        / LAG(monthly_sales) OVER (ORDER BY order_month) * 100, 1
    ) AS mom_growth_pct
FROM monthly_revenue
ORDER BY order_month
""").df()
q4


> **인사이트**: 증감률이 +61.6%(1월) → -69.4%(2월)로 널뛰기함 — 변동폭이 매우 커서 표본 기간(6개월)이 짧은 탓에 계절성이라 단정하기 어렵고, 특히 2월은 데이터의 마지막 달이라 월 전체가 다 채워지지 않았을 가능성도 함께 확인해야 함.

## Q5. 카테고리 내 상위 상품이 해당 카테고리 매출의 몇 %를 차지하는가? (CTE 2개 + `DENSE_RANK()` + `SUM() OVER (PARTITION BY ...)`)

- **사용**: CTE 2개, `DENSE_RANK()`, `SUM() OVER` 파티션 합계

In [ ]:
q5 = con.sql("""
WITH product_sales AS (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS total_revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
),
ranked AS (
    SELECT
        category,
        product_name,
        total_revenue,
        DENSE_RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS category_rank,
        SUM(total_revenue) OVER (PARTITION BY category) AS category_total
    FROM product_sales
)
SELECT
    category,
    product_name,
    total_revenue,
    category_rank,
    ROUND(total_revenue / category_total * 100, 1) AS revenue_share_pct
FROM ranked
WHERE category_rank <= 3
ORDER BY category, category_rank
""").df()
q5


> **인사이트**: Accessory는 판매 이력이 있는 상품이 양말 세트 하나뿐이라 점유율 100%(단일 상품 의존), Sandals는 51.0% vs 49.0%로 가장 균형 잡힌 반면 Boots·Running·Sneakers는 1위 상품이 60% 이상을 차지 — 후자 카테고리들은 대표 상품 품절/이슈 시 매출 타격이 클 수 있음.

## 종합 요약

| 질문 | 사용 기법 | 핵심 결과 |
|---|---|---|
| Q1. 월별 누적 매출 | CTE + `SUM() OVER` 누적 | 누적은 계속 증가, 월별로는 12월 급락 후 1월 반등 |
| Q2. 카테고리별 매출 TOP3 | CTE + `RANK()` | Sandals는 1·2위 격차 작음, 나머지는 1위 쏠림 |
| Q3. 고객별 재구매 간격 | CTE + `ROW_NUMBER()` + `LAG()` | 간격 9~122일로 편차 크고, 4명은 재구매 없음 |
| Q4. 월별 매출 증감률 | CTE + `LAG()` | -69.4% ~ +61.6%로 변동성 매우 큼 |
| Q5. 카테고리 내 매출 비중 | CTE 2개 + `DENSE_RANK()` + `SUM() OVER` | Sandals 균형(51/49), 나머지는 1위 상품 60%+ 집중 |

**조건 충족 체크**
- WITH(CTE) 2개 이상: 전 문항 사용, Q5는 CTE 2개 (요구 충족)
- 순위 함수: RANK(Q2), ROW_NUMBER(Q3), DENSE_RANK(Q5) (요구 1회 이상 충족)
- LAG/LEAD: Q3, Q4 (요구 1회 이상 충족)
- SUM() OVER 누적: Q1(누적), Q5(파티션 합계) (요구 1회 이상 충족)

**셀프 체크**
1. 왜 이렇게 작성했나 — 매출 집계 단위(월/카테고리/상품)가 질문 의도와 맞는지 CTE 단계별로 확인.
2. 어디서 틀릴 수 있는가 — `status`가 Cancelled/Returned인 주문도 order_items에 없다면 매출 계산에 영향 없음(이번 데이터는 취소/반품 건 order_items 미등록 확인됨), order_seq=1인 고객의 LAG 값이 NULL로 나오는 것은 의도된 결과.
3. 왜 신뢰할 수 있는가 — Q1과 Q4의 monthly_sales 값이 동일 CTE 로직으로 일치, Q2·Q5의 total_revenue 합계가 서로 같음을 확인.